# BSRNN workbook
The aim for this workbook is to start designing the components of the BSRNN from the 'High Fidelity Speech Enhancement BSRNN' paper.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import soundfile as sf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [6]:
class TrialDataset(torch.utils.data.Dataset):
    def __init__(self, manifest_csv, data_root, split, chunk_s, sample_rate, seed, random_crop=True):
        self.data_root    = Path(data_root)
        self.split        = split                   
        self.chunk_s      = chunk_s        
        self.sample_rate  = sample_rate
        self.seed         = seed
        self.random_crop  = random_crop
        self.chunk_frames = int(chunk_s * sample_rate)
        self.epoch        = 0                       

        self.manifest_df = pd.read_csv(manifest_csv)

    def __len__(self):
        return len(self.manifest_df)

    def __getitem__(self, idx):
        # there are a couple of things that I need to get:
        # 1. the mixture audio
        # 2. the target speaker audio
        # 3. the enrollment audio

        # Then there are a few things that are not necessary but are useful:
        # 1. the meta data
        # 2. the trial_id
        # 3. the crop absent

        row = self.manifest_df.iloc[idx]
        trial_directory = self.data_root / "rendered" / self.split / row["trial_id"]
        mixture_directory = trial_directory / "mixture.wav"
        number_of_frames = sf.info(str(mixture_directory)).frames

        start_offset = self._crop_offset_start(idx, number_of_frames)
        mixture_audio = self._read_in_wav(mixture_directory, start=start_offset, frames=self.chunk_frames)
        target_audio = self._read_in_wav(trial_directory / "target.wav", start=start_offset, frames=self.chunk_frames)
        enrollment_audio = self._read_in_wav(trial_directory / "enrollment.wav")

        crop_absent = bool(target_audio.abs().max() == 0)
        return {
            "mixture": mixture_audio,
            "target": target_audio,
            "enrollment": enrollment_audio,
            "crop_absent": crop_absent,
            "trial_id": str(row["trial_id"]),
            "meta": {
                "condition":        str(row["condition"]),
                "clip_absent":      bool(row["target_absent"]),
                "sir_db":           float(row["sir_db"]),
                "snr_db":           float(row["snr_db"]),
                "overlap_achieved": float(row["overlap_achieved"]),
                "regime":           str(row["regime"]),
                "same_gender":      float(row["same_gender"]),
            },
        }


    def _read_in_wav(self, path, start=0, frames=-1):
        # Read in a wav file and return a torch tensor of shape (frames,)
        with sf.SoundFile(str(path)) as f:
            assert f.samplerate == self.sample_rate, f"Sample rate mismatch: {f.samplerate} != {self.sample_rate}"
            assert f.channels == 1, f"Channel mismatch: {f.channels} != 1"

            if start > 0:
                f.seek(start)
            
            x = f.read(frames, dtype='float32', always_2d=False)

        return torch.from_numpy(np.ascontiguousarray(x)).to(device)

    def _crop_offset_start(self, idx, n_frames):
        # This is used to determine the starting point of the crop for the audio. It will return a random starting point if random_crop is True, otherwise it will return 0. This will be used when reading in the audio files to ensure that they are all the same length. The starting point will be determined by the seed, epoch, and index of the sample. This is to ensure that the same starting point is used for all audio files in a given epoch.

        max_start = n_frames - self.chunk_frames
        assert max_start >= 0, f"clip {n_frames} shorter than chunk {self.chunk_frames}"

        epoch = self.epoch if self.random_crop else 0
        rng = np.random.default_rng((self.seed, epoch, idx))
        return int(rng.integers(0, max_start + 1))

    def set_epoch(self, epoch):
        # call this at the top of each training epoch to ensure that the random cropping is consistent across all samples in the dataset. This is important for reproducibility and to ensure that the model sees the same data in each epoch.
        self.epoch = epoch


In [21]:
smoke_train_dataset = TrialDataset(
    manifest_csv="./../../data/manifests/smoke_train.csv",
    data_root="./../../data",
    split="smoke_train",
    chunk_s=4.0, # follow CARTSE
    sample_rate=16000,
    seed=42
)

smoke_valid_dataset = TrialDataset(
    manifest_csv="./../../data/manifests/smoke_val.csv",
    data_root="./../../data",
    split="smoke_valid",
    chunk_s=4.0, # follow CARTSE
    sample_rate=16000,
    seed=42,
    random_crop=False,
)

smoke_train_loader = torch.utils.data.DataLoader(
    smoke_train_dataset,
    batch_size=12, # Follow CARTSE
    shuffle=True,
    num_workers=4
)

smoke_valid_loader = torch.utils.data.DataLoader(
    smoke_valid_dataset,
    batch_size=12, # Follow CARTSE
    shuffle=False,
    num_workers=4
)

In [22]:
# Example of how to use the dataset and dataloader
for batch in smoke_train_loader:
    mixture = batch["mixture"]
    target = batch["target"]
    enrollment = batch["enrollment"]
    crop_absent = batch["crop_absent"]
    trial_id = batch["trial_id"]
    meta = batch["meta"]

    print(f"Mixture shape: {mixture.shape}")
    print(f"Target shape: {target.shape}")
    print(f"Enrollment shape: {enrollment.shape}")
    print(f"Crop absent: {crop_absent}")
    print(f"Trial ID: {trial_id}")
    print(f"Meta: {meta}")
    break  # Just process one batch for demonstration 

Mixture shape: torch.Size([12, 64000])
Target shape: torch.Size([12, 64000])
Enrollment shape: torch.Size([12, 80000])
Crop absent: tensor([False, False,  True, False, False, False,  True,  True, False, False,
        False, False])
Trial ID: ['smoke_train-42-000011', 'smoke_train-42-000049', 'smoke_train-42-000003', 'smoke_train-42-000030', 'smoke_train-42-000026', 'smoke_train-42-000004', 'smoke_train-42-000040', 'smoke_train-42-000009', 'smoke_train-42-000048', 'smoke_train-42-000008', 'smoke_train-42-000020', 'smoke_train-42-000042']
Meta: {'condition': ['both', 'target_only', 'noise_only', 'both', 'target_only', 'both', 'interferer_only', 'interferer_only', 'both', 'target_only', 'both', 'both'], 'clip_absent': tensor([False, False,  True, False, False, False,  True,  True, False, False,
        False, False]), 'sir_db': tensor([7.3200,    nan,    nan, 3.6400,    nan, 1.2100,    nan,    nan, 8.0300,
           nan, 3.4400, 8.9600], dtype=torch.float64), 'snr_db': tensor([19.2700, 

## 1. Create the STFT

## 2. Build the band plan table as a pure function
